# What drives Airbnb pricing in Toronto?

1. Which neighbourhood has the highest average price per night?
2. Correlation between review scores rating and price.
3. Correlation between bedrooms and price.
4. Correlation between amenetites and price.
5. Correlation between availabilty_365 and price.

In [93]:
import pandas as pd

df = pd.read_csv('listings_clean.csv', low_memory=False)

df.shape

(14427, 77)

## Identify key columns

In [94]:
key_df = df[['host_name', 'neighbourhood_cleansed', 'bedrooms', 'review_scores_rating', 'amenities', 'price', 'availability_365']]

## 1. Which neighbourhood has the highest average price per night?

### Group neighbourhoods by listing count and average price, min price and max price

In [95]:
neighbourhood_df = df.groupby('neighbourhood_cleansed').agg(
                        listings_count=('host_name', 'count'),
                        avg_price=('price','mean'),
                        min_price=('price','min'),
                        max_price=('price','max')
                    ).sort_values('avg_price',ascending=False).round(2)

## 2. Correlation between review score rating and price

### Hypothese: positive correlation

Reasoning: people would pay more for a place with better review than a poor one.

Check missing values in review_scores_rating column


In [96]:
print(key_df['review_scores_rating'].isnull().sum())

2951


In [97]:
clean_key_df = key_df[~key_df['review_scores_rating'].isnull()]

Calculate average review score rating

In [98]:
clean_key_df['review_scores_rating'].mean()

4.801362844196585

In [99]:
clean_key_df[['review_scores_rating', 'price']].corr(method='pearson')

,review_scores_rating,price
review_scores_rating,1.000000,0.044398
price,0.044398,1.000000


Review score rating shows a very weak correlation of 0.04 with price. This suggests that review scores do not drive Airbnb pricing in Toronto. This is likely because the average review score rating is 4.8, which suggests that most listings cluster around the highest score (5.0) with very little spread, and is heavily skewed.

Airbnb hosts should not factor in review score rating when pricing their Airbnb property.

## 3. Correlation between bedrooms and price

### Hypothesis: positive

Reasoning: more rooms means bigger space, which is a common pricing factor in properties

In [100]:
print(clean_key_df['bedrooms'].isnull().sum())

clean_key_df = clean_key_df[~clean_key_df['bedrooms'].isnull()]

4


In [101]:
clean_key_df[['bedrooms', 'price']].corr(method='pearson')

,bedrooms,price
bedrooms,1.000000,0.269677
price,0.269677,1.000000


Bedrooms shows a moderate positive correlation of 0.27 with price, suggesting that larger properties tend to command higher prices. However, this relationship is likely moderated by neighbourhood — a 1-bedroom in a high-demand area may price higher than a 3-bedroom in a less desirable location. Bedrooms alone is not a strong standalone pricing driver.

## 4. Correlation between amenities and price

In [102]:
import ast

In [103]:
clean_key_df['amenities_count'] = clean_key_df['amenities'].apply(lambda x: len(ast.literal_eval(str(x))))

In [104]:
print(clean_key_df['amenities_count'].describe())

count    11472.000000
mean        40.375610
std         13.858572
min          1.000000
25%         32.000000
50%         41.000000
75%         49.000000
max        103.000000
Name: amenities_count, dtype: float64


In [107]:
print(clean_key_df['amenities_count'].isnull().sum())

0


In [108]:
clean_key_df[['amenities_count', 'price']].corr(method='pearson')

,amenities_count,price
amenities_count,1.000000,0.119348
price,0.119348,1.000000


Amenities count shows a weak positive correlation of 0.12 with price. While more amenities generally trend toward higher prices, neighbourhood appears to be a stronger pricing driver — a listing in a premium neighbourhood with fewer amenities may still command a higher price than one with extensive amenities in a less desirable area.

## 5. Correlation between availability_365 and price

In [110]:
clean_key_df[['availability_365', 'price']].corr(method='pearson')

,availability_365,price
availability_365,1.000000,-0.002583
price,-0.002583,1.000000


Availability_365 shows weak negative -0.0025 with price, which suggessts that availability_365 has no effect driving price. This occurs due to the possiblity that overpriced listings are sitting empty, hosts limiting availability, new listings not yet book, etc. These cancel out each other, and result in low correlations. 

## Summary

Across five variables examined — neighbourhood, review scores, bedrooms, amenities, and availability — neighbourhood is the strongest driver of Airbnb pricing in Toronto. Review scores, amenities count, and availability show weak to no correlation with price, suggesting that where a listing is located matters far more than how it's rated or what it offers.